# Figures of the paper (LLM Coverage, release 20261105)

Candidate figures for Sections 5.1 to 5.4, about two per research question. The paper uses RQ1-B (`rq1_family_flow`), RQ2-B
(`rq2_rank_bump`), RQ3-C (`rq3_swaps`), and RQ4 (`rq4_behavior`). All are drawn at the final single-column width of the Springer template
(5.1 inches) with the paper's Computer Modern font, so font sizes in the PDF are the real ones and the figures go in with
`\includegraphics[width=\linewidth]{...}` without rescaling.

- Output: `figures/<release>/<name>.pdf` (vector, for LaTeX) and `.png` (300 dpi, for checking).
- Data: only files of the release in `deliverables/` and `outputs/`, plus the population cache for the family masses.
- To run: `python run_all_experiments.py --only paper_figures` (headless, about 20 s), or open it with the project's kernel and run all cells.
- Colors: a validated categorical palette (fixed order blue, orange, aqua, yellow, magenta, green) and a diverging blue to red scale with a
  neutral gray midpoint. Every figure has a legend or direct labels, so color is never the only cue.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

RELEASE = os.environ.get("DMKD_RELEASE", "20261105")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DELIV = ROOT / "deliverables" / RELEASE
OUTP = ROOT / "outputs" / RELEASE
FIG = ROOT / "figures" / RELEASE
FIG.mkdir(parents=True, exist_ok=True)
os.environ["DMKD_RELEASE"] = RELEASE
for p in (ROOT / "src", ROOT, ROOT / "analysis"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

W = 5.1  # text width of the single-column Springer template, in inches
INK, MUTED, FAINT, RULE, GRID = "#0b0b0b", "#52514e", "#9a9893", "#b5b4ae", "#ecebe7"
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100", magenta="#e87ba4", green="#008300",
         red="#e34948", gray="#c9c8c2")
RAMP = {250: "#86b6ef", 450: "#2a78d6", 650: "#104281"}           # ordinal blue ramp
DIVERGING = LinearSegmentedColormap.from_list("red_gray_blue", [C["red"], "#a8a7a1", C["blue"]])

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["cmr10"], "mathtext.fontset": "cm",
    "axes.unicode_minus": False, "axes.formatter.use_mathtext": True,
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "legend.frameon": False, "legend.handlelength": 1.4,
    "axes.edgecolor": RULE, "axes.linewidth": 0.6, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.5, "figure.dpi": 150,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
})


def save(fig, name):
    fig.savefig(FIG / f"{name}.pdf")
    fig.savefig(FIG / f"{name}.png", dpi=300)


def swarm(values, diameter, unit_per_inch_y, unit_per_inch_x):
    # Horizontal beeswarm offsets: each point moves to the smallest offset without overlap.
    y = np.asarray(values, float) / unit_per_inch_y
    order = np.argsort(y)
    placed, offsets = [], np.zeros(len(y))
    for i in order:
        for k in range(0, 200):
            for cand in ((0.0,) if k == 0 else (k * diameter * 0.5, -k * diameter * 0.5)):
                if all((cand - ox) ** 2 + (y[i] - oy) ** 2 >= diameter ** 2 for ox, oy in placed):
                    offsets[i] = cand
                    placed.append((cand, y[i]))
                    break
            else:
                continue
            break
    return offsets * unit_per_inch_x


def ribbon(ax, top, bottom, y_top, y_bottom, color, alpha=0.30):
    # Bezier ribbon between segment `top` (x0, x1) at y_top and segment `bottom` at y_bottom.
    (a0, a1), (b0, b1) = top, bottom
    ym = (y_top + y_bottom) / 2
    verts = [(a0, y_top), (a0, ym), (b0, ym), (b0, y_bottom), (b1, y_bottom), (b1, ym), (a1, ym), (a1, y_top), (a0, y_top)]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.LINETO,
             MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha, zorder=1))


def sigmoid_path(y0, y1, x0, x1, steps=40, k=10.0):
    t = np.linspace(0, 1, steps)
    s = 1 / (1 + np.exp(-k * (t - 0.5)))
    s = (s - s[0]) / (s[-1] - s[0])
    return x0 + (x1 - x0) * t, y0 + (y1 - y0) * s

In [ ]:
c4 = pd.read_csv(OUTP / "appendix_c" / "appendix_c4_complete_audit_per_paper.csv")
bands = pd.read_csv(DELIV / "RQ1_audit" / "reference_bands.csv")
sens = pd.read_csv(DELIV / "RQ2_sensitivity" / "sensitivity_summary.csv")
sens_pp = pd.read_csv(DELIV / "RQ2_sensitivity" / "sensitivity_by_paper.csv")
ladder = pd.read_csv(DELIV / "F_feasibility" / "feasibility_ladder.csv")
trade = pd.read_csv(DELIV / "RQ3_construction" / "tradeoff_by_paper.csv")
swaps = pd.read_csv(DELIV / "RQ3_construction" / "swaps_by_paper.csv")
assoc = pd.read_csv(DELIV / "P1_1_validation" / "gap_error_association.csv")
robust = pd.read_csv(DELIV / "P1_1_validation" / "robustness_summary.csv")
bounds = pd.read_csv(DELIV / "P1_4_bound" / "simulated_panel_bounds.csv")
print(len(c4), "papers;", len(bounds), "simulated panel-outcome bounds")

Mean mass by family over the 46 papers, each at its own cutoff: panel mass ($\sigma_P$), the download-based adoption target ($\alpha=0.10$),
and a likes-based target. It uses the population cache and the functions of analyses P1.2 and RQ1, so the numbers match
`RQ1_audit/family_composition.csv`.

In [ ]:
import common, rq_common as rq, p1_2_target_sensitivity as p12

F = common.foundation()
contexts = rq.load_contexts()
scores, definitions = p12.build_scores(F), p12.target_definitions(F)
families = F.variant["family_id"].astype(str).to_numpy()
per_paper = []
for pid in rq.scoreable(F):
    eligible = np.asarray(contexts[pid]["eligible"], bool)
    per_paper.append(pd.DataFrame({
        "family": families,
        "Panels": F.sigma(F.panels[pid]["entries"]),
        "Downloads": p12.make_target(definitions["alpha_0.10"], scores, eligible),
        "Likes": p12.make_target(definitions["likes"], scores, eligible),
    }).groupby("family").sum())
shares = sum(per_paper) / len(per_paper)
shares.sort_values("Panels", ascending=False).head(8).round(3)

## RQ1: coverage of existing panels

### Figure RQ1-A: percentiles against reference panels (ECDF)

If published panels were random draws of the same size, each one's percentile would be uniform and the cumulative curve would follow the
dashed diagonal. Against uniform references (left), the gap follows the diagonal: fidelity to adoption is that of a random draw. Breadth and
structural imbalance rise right at zero: panels are narrower than almost every draw. Against adoption-guided references (right), all three
curves rise at zero.

*Suggested caption:* Cumulative distribution of the 46 observed panels' percentiles among 120 same-size reference panels drawn uniformly (left)
or in proportion to adoption (right). If panels were random draws, curves would follow the dashed diagonal; curves above it mean panels do
worse than the references.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W, 2.35), sharey=True, gridspec_kw={"wspace": 0.12})
coords = [("adoption", "Adoption gap $G_a$", C["blue"]),
          ("breadth", "Lineage breadth $B$", C["orange"]),
          ("structural", r"Structural imbalance $D_\infty$", C["aqua"])]
for ax, (ref, title) in zip(axes, [("uniform", "vs. uniform references"), ("adoption_weighted", "vs. adoption-weighted references")]):
    ax.plot([0, 1], [0, 1], color=FAINT, lw=0.8, ls=(0, (3, 2)), zorder=1)
    grid = np.linspace(0, 1, 601)
    for key, label, color in coords:
        v = np.sort(c4[f"{ref}_{key}_percentile"].to_numpy())
        ecdf = np.searchsorted(v, grid, side="right") / len(v)
        ax.plot(grid, ecdf, color=color, lw=1.5, label=label, zorder=3, drawstyle="steps-post")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_title(title, color=MUTED, pad=4)
    ax.grid(True, axis="both")
    ax.set_axisbelow(True)
    ax.text(0.80, 0.735, "random panels", color=FAINT, fontsize=6.6, rotation=45, rotation_mode="anchor",
            transform_rotates_text=True, ha="center", va="bottom")
fig.supxlabel("Percentile of the observed panel among 120 same-size reference panels", fontsize=8, y=-0.03)
axes[0].set_ylabel("Share of papers")
axes[0].text(0.97, 0.04, "curves above the diagonal:\npanels do worse than\nthe reference panels", fontsize=6.6,
             color=MUTED, ha="right", va="bottom", linespacing=1.0)
axes[1].legend(loc="lower right", handlelength=1.6)
save(fig, "rq1_percentile_ecdf")
plt.show()

### Figure RQ1-B: where the mass goes (flow between target and panels)

Three stacked bars with the same family order: the download-based adoption target, the panels' mass, and the likes-based target. Ribbons
connect the same family across bars. Downloads sit on GPT-2 and T5, which almost no panel evaluates; panels sit on Llama and DeepSeek; and the
likes-based target looks much more like the panels. The figure serves RQ1 (why fidelity to adoption is low) and RQ2 (why the target changes
the conclusion).

*Suggested caption:* Mean share of mass by family, over the 46 papers at their cutoffs, in the download-based adoption target, in the evaluated
panels, and in a likes-based target. Ribbons connect the same family.

In [ ]:
order = ["llama", "qwen", "deepseek", "mistral", "gpt2", "t5"]
names = {"llama": "Llama", "qwen": "Qwen", "deepseek": "DeepSeek", "mistral": "Mistral", "gpt2": "GPT-2", "t5": "T5", "other": "Other"}
colors = {"llama": C["blue"], "qwen": C["orange"], "deepseek": C["aqua"], "mistral": C["yellow"], "gpt2": C["magenta"],
          "t5": C["green"], "other": C["gray"]}
rows = [("Downloads", "Adoption target\n(downloads)"), ("Panels", "Evaluation\npanels"), ("Likes", "Adoption target\n(likes)")]
table = shares.loc[order, [r[0] for r in rows]]
table.loc["other"] = 1 - table.sum()
fams = order + ["other"]

fig, ax = plt.subplots(figsize=(W, 2.25))
ys = {"Downloads": 2.0, "Panels": 1.0, "Likes": 0.0}
h = 0.26
edges = {}
for key, label in rows:
    left = 0.0
    for f in fams:
        width = float(table.loc[f, key])
        ax.barh(ys[key], width, left=left, height=h, color=colors[f], edgecolor="white", linewidth=0.8, zorder=3)
        edges[(key, f)] = (left, left + width)
        if width > 0.075:
            ax.text(left + width / 2, ys[key], f"{names[f]}\n{width:.0%}", ha="center", va="center",
                    fontsize=6.3, color="white" if f in ("llama", "green", "t5") else INK, zorder=4, linespacing=0.95)
        left += width
    ax.text(-0.015, ys[key], label, ha="right", va="center", fontsize=7.3, color=INK, linespacing=1.0)
for upper, lower in [("Downloads", "Panels"), ("Panels", "Likes")]:
    for f in fams:
        ribbon(ax, edges[(upper, f)], edges[(lower, f)], ys[upper] - h / 2, ys[lower] + h / 2, colors[f], alpha=0.28)
ax.set_xlim(0, 1)
ax.set_ylim(-0.25, 2.25)
ax.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[f]) for f in fams]
ax.legend(handles, [names[f] for f in fams], ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.0), handlelength=1.0,
          columnspacing=1.0, handletextpad=0.4)
save(fig, "rq1_family_flow")
plt.show()

### Figure RQ1-C: each paper against the median random panel (waterfall)

Each column is a paper, sorted by the difference between its observed gap and that of the median random panel of the same size. The gray band
is the 5% to 95% range of uniform panels, the orange band that of adoption-guided panels, and the blue dot the observed panel. About half
of the papers sit above zero (worse than the median draw), and almost none reach the orange band.

*Suggested caption:* Adoption gap of each observed panel relative to the median of 120 same-size uniform reference panels, with the 5-95% range
of uniform (gray) and adoption-weighted (orange) references. Papers are sorted by relative gap.

In [ ]:
b = bands.copy()
b["rel"] = b["observed_adoption_gap"] - b["uniform_p50"]
b = b.sort_values("rel").reset_index(drop=True)
x = np.arange(len(b))
fig, ax = plt.subplots(figsize=(W, 2.15))
ax.axhline(0, color=RULE, lw=0.7, zorder=1)
ax.vlines(x, b["uniform_p05"] - b["uniform_p50"], b["uniform_p95"] - b["uniform_p50"], color=FAINT, lw=2.6, alpha=0.45,
          zorder=2, capstyle="round", label="Uniform references, 5-95%")
ax.vlines(x, b["adoption_p05"] - b["uniform_p50"], b["adoption_p95"] - b["uniform_p50"], color=C["orange"], lw=1.3,
          alpha=0.85, zorder=2, capstyle="round", label="Adoption-weighted references, 5-95%")
better = b["rel"] < 0
ax.scatter(x[better], b.loc[better, "rel"], s=13, color=C["blue"], edgecolor="white", linewidth=0.5, zorder=4, label="Observed panel")
ax.scatter(x[~better], b.loc[~better, "rel"], s=13, facecolor="white", edgecolor=C["blue"], linewidth=0.9, zorder=4,
           label="Observed panel, worse than median reference")
ax.set_xlim(-1, len(b))
ax.set_xticks([])
ax.set_xlabel("Papers, sorted by observed gap relative to the median uniform panel")
ax.set_ylabel(r"$G_a$ minus median" + "\nuniform reference")
ax.set_ylim(-1.0, 0.34)
ax.text(0, -0.97, r"$\leftarrow$ better than the median random panel", fontsize=6.8, color=MUTED, va="bottom")
ax.text(len(b) - 1, -0.97, r"worse $\rightarrow$", fontsize=6.8, color=MUTED, va="bottom", ha="right")
ax.grid(True, axis="y")
ax.set_axisbelow(True)
ax.spines["bottom"].set_visible(False)
ax.legend(ncol=2, loc="upper left", bbox_to_anchor=(0, 1.22), columnspacing=1.2)
save(fig, "rq1_waterfall")
plt.show()

## RQ2: sensitivity to modelling choices

### Figure RQ2-A: robustness map

Each point is one modelling alternative. The x axis is the stability of the paper ranking (Kendall $\tau$ with the primary specification); the
y axis is the conclusion (median percentile of the gap against uniform references). Measurement choices (topological geometry, resolution,
cutoff) sit on the right, close to the primary specification. Target and population spread the y axis from 0.12 to 0.99: the conclusion
about adoption depends on what is declared. The right panel shows, on the same scale, that breadth and imbalance stay near zero under every
alternative: narrowness does not depend on these choices.

*Suggested caption:* Each point is one alternative modelling choice. Left: Kendall correlation of paper rankings with the primary specification
against the median percentile of observed adoption gaps among uniform reference panels. Right: the same percentile for breadth and structural
imbalance, on the same scale, for the alternatives that change them.

In [ ]:
cat = {"target": ("Target", C["blue"], "o"), "geometry": ("Geometry", C["orange"], "s"), "resolution": ("Resolution", C["aqua"], "D"),
       "cutoff": ("Cutoff", C["yellow"], "^"), "population": ("Population", C["magenta"], "v")}
s = sens[sens["papers"] >= 41].copy()
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(W, 2.6), sharey=True, gridspec_kw={"width_ratios": [3.2, 1], "wspace": 0.08})
from scipy.spatial import ConvexHull
ax.axhline(0.5, color=FAINT, lw=0.8, ls=(0, (3, 2)), zorder=1)
ax.text(0.30, 0.515, "median random panel", fontsize=6.5, color=FAINT, va="bottom")
measure = s[s["choice"].isin(["primary", "resolution", "cutoff"]) | (s["alternative"] == "topological")]
pts = measure[["kendall_tau", "median_uniform_percentile"]].to_numpy()
pts = np.vstack([pts, [[1.0, pts[:, 1][0]]]])
hull = pts[ConvexHull(pts).vertices]
center = hull.mean(axis=0)
hull = center + (hull - center) * 1.18
ax.fill(hull[:, 0], hull[:, 1], color=FAINT, alpha=0.13, lw=9, edgecolor=(0.6, 0.6, 0.58, 0.13), joinstyle="round", zorder=0)
ax.text(0.80, 0.665, "measurement choices:\ncutoff, resolution,\ntopological geometry", fontsize=6.4, color=MUTED, ha="center",
        va="bottom", linespacing=0.95)
for choice, (label, color, marker) in cat.items():
    part = s[s["choice"] == choice]
    ax.scatter(part["kendall_tau"], part["median_uniform_percentile"], s=26, color=color, marker=marker, edgecolor="white",
               linewidth=0.6, zorder=3, label=label)
prim = s[s["choice"] == "primary"].iloc[0]
ax.scatter([1.0], [prim["median_uniform_percentile"]], s=90, marker="*", color=INK, edgecolor="white", linewidth=0.6, zorder=4,
           label="Primary")
notes = {"likes": ("Likes target", (0.07, 0.90), "left"),
         "adoption_weighted_ablation": ("Geometries weighted by\nrepositories or adoption", (0.66, 0.93), "left"),
         "assistant_models": ("Assistant models", (0.40, 0.855), "left"),
         "claim_conditioned": ("Claim-conditioned", (0.30, 0.76), "right"),
         "large_models_ge_34b": (r"Models $\geq$34B", (0.11, 0.40), "left"),
         "uniform": ("Uniform and\nlog-download targets", (0.63, 0.04), "left"),
         "base_models": ("Base models", (0.86, 0.10), "left")}
for alt, (text, (tx, ty), ha) in notes.items():
    r = s[s["alternative"] == alt].iloc[0]
    ax.annotate(text, xy=(r["kendall_tau"], r["median_uniform_percentile"]), xytext=(tx, ty), fontsize=6.4, color=MUTED, ha=ha,
                va="center", arrowprops=dict(arrowstyle="-", color=RULE, lw=0.5, shrinkA=1, shrinkB=3), linespacing=0.95)
ax.set_xlim(0.0, 1.05)
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel(r"Ranking stability: Kendall $\tau$ with the primary ranking")
ax.set_ylabel("Median percentile of $G_a$\namong uniform references")
ax.set_title("Adoption fidelity", color=MUTED, pad=4)
ax.grid(True)
ax.set_axisbelow(True)

rng = np.random.default_rng(3)
sub = s[s["median_breadth_percentile"].notna()]
for j, col in enumerate(["median_breadth_percentile", "median_structural_percentile"]):
    for choice, (label, color, marker) in list(cat.items()) + [("primary", ("Primary", INK, "*"))]:
        part = sub[sub["choice"] == choice]
        jitter = rng.uniform(-0.18, 0.18, len(part))
        ax2.scatter(j + jitter, part[col], s=60 if marker == "*" else 22, color=color, marker=marker, edgecolor="white",
                    linewidth=0.5, zorder=3)
ax2.axhline(0.5, color=FAINT, lw=0.8, ls=(0, (3, 2)), zorder=1)
ax2.axhspan(0, 0.08, color=C["orange"], alpha=0.08, lw=0)
ax2.text(0.5, 0.13, r"all $\leq 0.08$", ha="center", fontsize=6.6, color=MUTED)
ax2.set_xticks([0, 1], ["$B$", r"$D_\infty$"])
ax2.set_xlim(-0.5, 1.5)
ax2.set_title("Narrowness", color=MUTED, pad=4)
ax2.grid(True, axis="y")
ax2.set_axisbelow(True)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.07), columnspacing=1.0, handletextpad=0.3)
save(fig, "rq2_robustness_map")
plt.show()

### Figure RQ2-B: paper ranks across specifications (bump chart)

Each line is a paper, and the columns go from the primary specification to the alternatives that change the question most, in decreasing
Kendall $\tau$. Color marks the paper's third in the primary ranking. With topological geometry or no cutoff, the colors stay separated; with the
claim-conditioned population or with likes, they mix: the ranking answers a different question.

*Suggested caption:* Rank of each paper by adoption gap (1 = closest to the target) under the primary specification and four alternatives,
ordered by decreasing Kendall correlation. Colors mark the paper's third in the primary ranking.

In [ ]:
specs = [("primary", "Primary"), ("topological", "Topological\ngeometry"), ("no_cutoff", "No cutoff"),
         ("claim_conditioned", "Claim-\nconditioned"), ("likes", "Likes\ntarget")]
ranks = pd.DataFrame({alt: sens_pp[sens_pp["alternative"] == alt].set_index("paper_id")["adoption_gap"].rank(method="first")
                      for alt, _ in specs}).dropna()
taus = sens.set_index("alternative")["kendall_tau"]
third = pd.cut(ranks["primary"], [0, 15.5, 31.5, 99], labels=[650, 450, 250]).astype(int)
fig, ax = plt.subplots(figsize=(W, 2.7))
for pid, row in ranks.iterrows():
    color = RAMP[third[pid]]
    for i in range(len(specs) - 1):
        xs, ys_ = sigmoid_path(row.iloc[i], row.iloc[i + 1], i, i + 1)
        ax.plot(xs, ys_, color=color, lw=0.75, alpha=0.8, solid_capstyle="round", zorder=2)
    ax.scatter(range(len(specs)), row.values, s=7, color=color, zorder=3, edgecolor="white", linewidth=0.3)
for i, (alt, label) in enumerate(specs):
    ax.text(i, -2.2, label, ha="center", va="bottom", fontsize=7, color=INK, linespacing=0.95)
    if alt != "primary":
        ax.text(i, 49.0, rf"$\tau = {taus[alt]:.2f}$", ha="center", va="top", fontsize=6.8, color=MUTED)
ax.set_ylim(49.5, -4.5)
ax.set_xlim(-0.35, len(specs) - 0.65)
ax.set_xticks([])
ax.set_yticks([1, 10, 20, 30, 40, 46])
ax.set_ylabel("Rank by adoption gap")
ax.spines["bottom"].set_visible(False)
for x0 in range(len(specs)):
    ax.axvline(x0, color=GRID, lw=0.6, zorder=1)
handles = [Line2D([0], [0], color=RAMP[k], lw=1.6) for k in (650, 450, 250)]
ax.legend(handles, ["Primary ranks 1-15", "16-31", "32-46"], ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.02),
          columnspacing=1.2)
save(fig, "rq2_rank_bump")
plt.show()

## RQ3: constructing panels under constraints

### Figure RQ3-A: constraint ladder

Each point is a paper, at the gap reduction of the adoption-greedy panel on each rung of the ladder. Color is the ratio between the parameters
of the alternative panel and those of the original panel, on a log scale: red means much smaller models, gray the same size. The large
reductions of the first two rungs come from replacing the evaluated models with tiny ones; once composition is preserved, the color turns gray
and the reduction drops. The line joins the medians, and the number above each column is the median size of the candidate pool.

*Suggested caption:* Gap reduction achieved by the adoption-greedy alternative for each paper as constraints accumulate. Color shows the
alternative's total parameters relative to the observed panel (log scale); the line joins medians, and numbers above give the median candidate
pool.

In [ ]:
rungs = ["historical", "core", "cost", "license", "context", "adoption_floor"]
rung_labels = ["Available\nby cutoff", "+ access, role,\nmax. size", "+ size-by-role\ncomposition", "+ license", "+ context\nlength",
               "+ adoption\nfloor"]
L = ladder[ladder["rung"].isin(rungs)].copy()
L["log_ratio"] = np.log10(L["adoption_cost_ratio"].clip(1e-3, 100))
fig, ax = plt.subplots(figsize=(W, 2.55))
fig.canvas.draw()
ax.set_xlim(-0.6, len(rungs) - 0.4)
ax.set_ylim(-12, 92)
pos = ax.get_position()
x_per_in = (len(rungs) - 0.4 + 0.6) / (pos.width * W)
y_per_in = (92 + 12) / (pos.height * 2.55)
norm = TwoSlopeNorm(vmin=-2, vcenter=0, vmax=1)
medians = []
for i, r in enumerate(rungs):
    part = L[L["rung"] == r]
    off = swarm(part["adoption_gap_reduction_pct"].to_numpy(), diameter=0.043, unit_per_inch_y=y_per_in, unit_per_inch_x=x_per_in)
    sc = ax.scatter(i + off, part["adoption_gap_reduction_pct"], c=part["log_ratio"], cmap=DIVERGING, norm=norm, s=9,
                    edgecolor="white", linewidth=0.3, zorder=3)
    medians.append(part["adoption_gap_reduction_pct"].median())
    ax.text(i, 95, f"{int(np.floor(part['pool_size'].median() + 0.5)):,}", ha="center", va="bottom", fontsize=6.6, color=MUTED)
ax.text(-0.62, 102, "median pool:", ha="left", va="bottom", fontsize=6.6, color=MUTED)
ax.plot(range(len(rungs)), medians, color=INK, lw=1.0, zorder=4)
ax.scatter(range(len(rungs)), medians, s=20, color="white", edgecolor=INK, linewidth=0.9, zorder=5)
for i, m in enumerate(medians):
    ax.text(i + 0.12, m + 3, f"{m:.1f}" + "%", fontsize=6.6, color=INK, zorder=6)
ax.axhline(0, color=RULE, lw=0.6)
ax.set_xticks(range(len(rungs)), rung_labels, fontsize=6.8)
ax.tick_params(axis="x", length=0)
ax.set_ylabel("Gap reduction (%)")
ax.grid(True, axis="y")
ax.set_axisbelow(True)
cax = fig.add_axes([pos.x1 + 0.015, pos.y0 + 0.05, 0.012, pos.height * 0.8])
cb = fig.colorbar(sc, cax=cax, ticks=[-2, -1, 0, 1])
cb.ax.set_yticklabels(["0.01", "0.1", "1", "10"], fontsize=6.5)
cb.outline.set_visible(False)
cb.set_label("Parameters,\nalternative / panel", fontsize=6.8, labelpad=3)
save(fig, "rq3_constraint_ladder")
plt.show()

### Figure RQ3-B: trade-off paths between gap and breadth

For each paper, the light line joins the panels built as the weight $\lambda$ on the gap goes from 1 (gap only) to 0 (breadth only); the dark
line is the median. The shaded quadrant is where gap and breadth improve together. With composition preserved (left), equal weight
($\lambda=0.5$) lands in the quadrant for almost every paper; with all constraints (right), the gains shrink but stay compatible.

*Suggested caption:* Gap reduction and breadth gain of constructed panels relative to each observed panel as the weight on the adoption gap goes
from 1 to 0 (thin lines: papers; thick line: medians). The shaded quadrant marks panels that improve both coordinates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(W, 2.45), sharey=True, gridspec_kw={"wspace": 0.08})
lam_order = [1.0, 0.9, 0.75, 0.5, 0.25, 0.1, 0.0]
for ax, rung, title in zip(axes, ["cost", "adoption_floor"], ["Size-by-role composition preserved", "All constraints"]):
    d = trade[trade["rung"] == rung]
    ax.fill_between([0, 100], 0, 2000, color=C["aqua"], alpha=0.08, lw=0, zorder=0)
    ax.axhline(0, color=RULE, lw=0.6, zorder=1)
    ax.axvline(0, color=RULE, lw=0.6, zorder=1)
    for pid, g in d.groupby("paper_id"):
        g = g.set_index("lambda").loc[lam_order]
        ax.plot(g["gap_reduction_pct"], g["breadth_gain_pct"], color=RAMP[250], lw=0.5, alpha=0.45, zorder=2)
    half = d[d["lambda"] == 0.5]
    both = (half["gap_reduction_pct"] > 0) & (half["breadth_gain_pct"] > 0)
    ax.scatter(half["gap_reduction_pct"], half["breadth_gain_pct"], s=7, color=RAMP[450], zorder=3, edgecolor="white", linewidth=0.3)
    med = d.groupby("lambda")[["gap_reduction_pct", "breadth_gain_pct"]].median().loc[lam_order]
    ax.plot(med["gap_reduction_pct"], med["breadth_gain_pct"], color=RAMP[650], lw=1.8, zorder=4)
    ax.scatter(med["gap_reduction_pct"], med["breadth_gain_pct"], s=16, color="white", edgecolor=RAMP[650], linewidth=1.0, zorder=5)
    for lam, (gx, gy) in med.iterrows():
        if lam in (1.0, 0.5, 0.0):
            ax.annotate(rf"$\lambda={lam:g}$", (gx, gy), xytext=(4, 5), textcoords="offset points", fontsize=6.6, color=RAMP[650])
    ax.set_yscale("symlog", linthresh=10, linscale=0.6)
    ax.set_yticks([-10, 0, 10, 100, 1000], ["-10", "0", "10", "100", "1000"])
    ax.set_ylim(-40, 2000)
    ax.set_xlim(-30, 60 if rung == "cost" else 35)
    ax.set_title(title, color=MUTED, pad=4)
    ax.set_xlabel("Gap reduction (%)")
    ax.text(0.97, 0.965, f"both improve at $\\lambda=0.5$:\n{both.sum()} of {len(half)} papers", transform=ax.transAxes,
            ha="right", va="top", fontsize=6.6, color=MUTED, zorder=6,
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.85))
    ax.grid(True)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Breadth gain (%, symlog scale)")
save(fig, "rq3_tradeoff_paths")
plt.show()

### Figure RQ3-C: a few swaps are enough

Gap reduction from replacing 0 to 3 models of the original panel, each replacement keeping the size and role of the replaced model. Light lines
are papers, thick lines the medians, and the dashed line marks the median reduction of rebuilding the whole panel. Under all constraints,
three swaps practically reach the full rebuild.

*Suggested caption:* Gap reduction from replacing up to three models of each observed panel, each replacement keeping the replaced model's size
bucket and role (thin lines: papers; thick lines: medians). Dashed lines mark the median reduction of a full greedy rebuild under the same
constraints.

In [ ]:
fig, ax = plt.subplots(figsize=(W, 2.3))
styles = {"cost": ("Composition preserved", C["blue"]), "adoption_floor": ("All constraints", C["orange"])}
full = ladder.set_index("rung").groupby(level=0)["adoption_gap_reduction_pct"].median()
for rung, (label, color) in styles.items():
    d = swaps[swaps["rung"] == rung]
    curves = []
    for pid, g in d.groupby("paper_id"):
        g = g.set_index("swaps")["G"].reindex(range(4)).ffill()
        red = 100 * (g.iloc[0] - g) / g.iloc[0]
        curves.append(red.to_numpy())
        ax.plot(range(4), red, color=color, lw=0.5, alpha=0.22, zorder=2)
    med = np.median(np.vstack(curves), axis=0)
    ax.plot(range(4), med, color=color, lw=2.0, zorder=4, label=label)
    ax.scatter(range(4), med, s=18, color="white", edgecolor=color, linewidth=1.0, zorder=5)
    ax.axhline(full[rung], color=color, lw=0.9, ls=(0, (4, 2)), zorder=3)
    ax.text(3.6, full[rung] + 0.6, f"full rebuild {full[rung]:.1f}" + "%", fontsize=6.5, color=color, va="bottom", ha="right")
    for k in range(1, 4):
        ax.text(k, med[k] + (1.0 if rung == "cost" else -1.0), f"{med[k]:.1f}" + "%", fontsize=6.5, color=color, ha="center",
                va="bottom" if rung == "cost" else "top")
ax.set_xticks(range(4), ["observed", "1 swap", "2 swaps", "3 swaps"])
ax.set_xlim(-0.15, 3.65)
ax.set_ylim(-1.5, 30)
ax.set_ylabel("Gap reduction (%)")
ax.grid(True, axis="y")
ax.set_axisbelow(True)
ax.legend(loc="upper left")
save(fig, "rq3_swaps")
plt.show()

## Behavioral interpretation (not used in the paper; superseded by RQ4)

### Figure B-A: association with error and predictive gain

Left: rank correlation between gap and recovery error within each panel size, by benchmark, with family-cluster intervals. Right: out-of-sample
$R^2$ gain of each coordinate over the baseline, with panel sizes held out.

*Suggested caption:* Left: within-size Spearman correlation between adoption gap and recovery error, with family-cluster 95% intervals, for all
panels and for panels drawn from disjoint families. Right: gain in out-of-sample $R^2$ from adding each coordinate to the baseline, panel sizes
held out, with family-cluster 95% intervals.

In [ ]:
outcomes = [("IFEval", "IFEval"), ("BBH", "BBH"), ("MATH Lvl 5", "MATH L5"), ("GPQA", "GPQA"), ("MUSR", "MuSR"),
            ("MMLU-PRO", "MMLU-Pro"), ("salad_base_safety", "SALAD base"), ("salad_attack_safety", "SALAD attack")]
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(W, 2.45), gridspec_kw={"width_ratios": [1.15, 1], "wspace": 0.55})
for k, (panels, color, marker, face, dy) in enumerate([("all_models", C["blue"], "o", C["blue"], -0.14),
                                                        ("disjoint_families", C["orange"], "D", "white", 0.14)]):
    for i, (key, label) in enumerate(outcomes):
        r = assoc[(assoc["outcome"] == key) & (assoc["panels"] == panels)].iloc[0]
        ax.plot([r["family_low"], r["family_high"]], [i + dy] * 2, color=color, lw=1.0, alpha=0.8, solid_capstyle="round")
        ax.scatter(r["within_size_spearman"], i + dy, s=16, marker=marker, facecolor=face, edgecolor=color, linewidth=0.9,
                   zorder=3, label=("All panels" if panels == "all_models" else "Disjoint families") if i == 0 else None)
ax.axvline(0, color=RULE, lw=0.7)
ax.axhline(5.5, color=GRID, lw=0.6)
ax.set_yticks(range(len(outcomes)), [o[1] for o in outcomes])
ax.set_ylim(len(outcomes) - 0.4, -0.6)
ax.set_xlim(-0.3, 0.95)
ax.set_xlabel(r"Spearman $\rho$, gap vs. error")
ax.set_title("Association within panel size", color=MUTED, pad=4)
ax.legend(loc="upper center", bbox_to_anchor=(0.42, -0.2), ncol=2, handletextpad=0.2, columnspacing=1.0, fontsize=6.8)
ax.grid(True, axis="x")
ax.set_axisbelow(True)

coords = [("adoption_gap", "Adoption gap $G_a$"), ("structural_linf", r"Structural $D_\infty$"), ("breadth", "Breadth $B$"),
          ("coverage_q1", "$C(1)$"), ("coverage_q0", "$C(0)$"), ("full_profile", "Full profile")]
rs = robust[(robust["scope"] == "all") & (robust["scheme"] == "size")].set_index("predictor")
for i, (key, label) in enumerate(coords):
    r = rs.loc[key]
    color = C["blue"] if key == "adoption_gap" else (C["aqua"] if key == "structural_linf" else FAINT)
    ax2.plot([r["mean_delta_r2_low"], r["mean_delta_r2_high"]], [i, i], color=color, lw=1.4, solid_capstyle="round")
    ax2.scatter(r["mean_delta_r2"], i, s=20, color=color, edgecolor="white", linewidth=0.5, zorder=3)
    ax2.text(0.29, i, f"{int(r['benchmarks_positive'])}/8", fontsize=6.4, color=MUTED, va="center", ha="left")
ax2.text(0.29, -0.85, "positive", fontsize=6.4, color=MUTED, ha="left", va="center")
ax2.axvline(0, color=RULE, lw=0.7)
ax2.set_yticks(range(len(coords)), [c[1] for c in coords])
ax2.set_ylim(len(coords) - 0.4, -1.1)
ax2.set_xlim(-0.6, 0.27)
ax2.set_xlabel(r"$\Delta R^2_{\mathrm{OOS}}$ over baseline")
ax2.set_title("Incremental prediction", color=MUTED, pad=4)
ax2.grid(True, axis="x")
ax2.set_axisbelow(True)
save(fig, "behav_association_prediction")
plt.show()

### Figure B-B: how loose the theoretical bound is (ridgeline)

Distribution, by benchmark, of the ratio between the bound of Theorem 1 and the realized error over simulated panels, on a log scale. A vertical
line at 1 would be an exact bound.

*Suggested caption:* Ratio of the mean-recovery bound to the realized error over simulated panels, by outcome (log scale). A ratio of one would
be a tight bound; numbers give the median ratio.

In [ ]:
names_b = dict(outcomes)
b = bounds[bounds["ratio"] > 0].copy()
b["log_slack"] = -np.log10(b["ratio"])
order_b = b.groupby("outcome")["log_slack"].median().sort_values().index.tolist()
grid = np.linspace(-0.2, 3.6, 400)
fig, ax = plt.subplots(figsize=(W, 2.5))
step, scale = 0.62, 1.15
for i, key in enumerate(order_b):
    v = b.loc[b["outcome"] == key, "log_slack"].to_numpy()
    dens = gaussian_kde(v, bw_method=0.25)(grid)
    dens = dens / dens.max() * scale * step
    base = i * step
    color = C["orange"] if key.startswith("salad") else C["blue"]
    ax.fill_between(grid, base, base + dens, color=color, alpha=0.55, lw=0, zorder=10 - i)
    ax.plot(grid, base + dens, color="white", lw=1.0, zorder=10 - i)
    ax.plot(grid, base + dens, color=color, lw=0.6, zorder=10 - i)
    med = np.median(v)
    ax.plot([med, med], [base, base + np.interp(med, grid, dens)], color=INK, lw=0.8, zorder=11 - i)
    ax.text(-0.25, base + 0.05, names_b[key], ha="right", va="bottom", fontsize=7, color=INK)
    ax.text(3.62, base + 0.05, rf"${10 ** med:.0f}\times$", ha="right", va="bottom", fontsize=6.6, color=MUTED)
ax.axvline(0, color=INK, lw=0.8, ls=(0, (3, 2)), zorder=20)
ax.text(0.04, len(order_b) * step + 0.35, "tight bound", fontsize=6.6, color=INK, va="bottom")
ax.text(3.62, len(order_b) * step + 0.35, "median", fontsize=6.6, color=MUTED, ha="right", va="bottom")
ax.set_xticks([0, 1, 2, 3], [r"$1\times$", r"$10\times$", r"$100\times$", r"$1000\times$"])
ax.set_xlim(-0.2, 3.65)
ax.set_ylim(-0.05, len(order_b) * step + 0.75)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_xlabel("Bound / realized error (log scale)")
handles = [plt.Rectangle((0, 0), 1, 1, color=C["blue"], alpha=0.55), plt.Rectangle((0, 0), 1, 1, color=C["orange"], alpha=0.55)]
ax.legend(handles, ["Capability", "Safety"], loc="upper center", bbox_to_anchor=(0.55, 1.06), ncol=2)
save(fig, "behav_bound_slack")
plt.show()

## RQ4: behavioral validity of lineage coverage

### Figure RQ4: relatives behave alike, and panels spread over the tree recover means better

Data: `deliverables/<release>/RQ4_behavioral/` (step `rq4`, `analysis/rq4_behavioral.py`): variants with the six Open LLM Leaderboard
capability scores, which the framework never uses.

Left: semivariance of standardized scores by lineage relation (deepest common node), divided by that of random pairs; 1 means relatives are no
more alike than unrelated models. Gray boxes are 95% intervals from 999 permutations.

Right: each point is an audited paper, in the world of scored variants available at its cutoff and with its panel size. The designs give every
model the same inclusion probability (systematic sampling proportional to the adoption target) and differ only in the order: random
(lineage-blind, the dashed line), depth-first walk of the tree (lineage), size bucket (scale), and both. The ratio is the RMSE of the panel mean
against the blind design, geometric mean over the six benchmarks.

*Suggested caption:* Behavioral validity of lineage coverage on 624 variants with six Open LLM Leaderboard capability scores. (a) Mean
semivariance of standardized scores for pairs of variants, by their closest common node, relative to random pairs. Large markers are means over
benchmarks and small dots single benchmarks; gray boxes are 95% permutation intervals. (b) RMSE of panel means relative to the lineage-blind
design with the same inclusion probabilities, under the adoption target, for the 44 audited papers (dots); bars are medians. Dashed lines mark
random pairs (a) and the lineage-blind design (b).

In [ ]:
semi = pd.read_csv(DELIV / "RQ4_behavioral" / "lineage_semivariance.csv")
by_paper = pd.read_csv(DELIV / "RQ4_behavioral" / "designs_by_paper.csv")
paper_sum = pd.read_csv(DELIV / "RQ4_behavioral" / "designs_paper_summary.csv")

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(W, 2.45), gridspec_kw={"width_ratios": [1.12, 1], "wspace": 0.42})

# (a) semivariance by lineage relation, relative to random pairs
rels = ["same base", "same series", "same family", "different family"]
rel_labels = ["Same\nbase", "Same\nseries", "Same\nfamily", "Different\nfamily"]
styles = [("raw", "Benchmark scores", C["blue"], -0.13, True),
          ("scale-adjusted", "Scale-adjusted scores", C["orange"], 0.13, False)]
for x, rel in enumerate(rels):
    s = semi[(semi["relation"] == rel) & (semi["benchmark"] != "mean")]
    lo = (s["null_p025"] / s["null_mean"]).min()
    hi = (s["null_p975"] / s["null_mean"]).max()
    ax.add_patch(plt.Rectangle((x - 0.36, lo), 0.72, hi - lo, color=FAINT, alpha=0.22, lw=0, zorder=1))
    for scores, label, color, dx, filled in styles:
        part = semi[(semi["relation"] == rel) & (semi["scores"] == scores)]
        b = part[part["benchmark"] != "mean"]
        ax.scatter(np.full(len(b), x + dx), b["semivariance"] / b["null_mean"], s=5, color=color, alpha=0.45, lw=0, zorder=2)
        m = part[part["benchmark"] == "mean"].iloc[0]
        ax.scatter([x + dx], [m["semivariance"] / m["null_mean"]], s=30, zorder=4, linewidth=1.0,
                   color=color if filled else "white", edgecolor=color if not filled else "white",
                   label=label if x == 0 else None)
ax.axhline(1.0, color=FAINT, lw=0.8, ls=(0, (3, 2)), zorder=1)
ax.set_xticks(range(len(rels)))
ax.set_xticklabels(rel_labels, linespacing=0.95)
ax.tick_params(axis="x", length=0, pad=2)
ax.set_xlim(-0.55, len(rels) - 0.45)
ax.set_ylim(0, 1.22)
ax.set_ylabel("Semivariance relative\nto random pairs")
ax.set_title("(a) Lineage relatives behave alike", color=INK, pad=4)
ax.grid(True, axis="y")
ax.set_axisbelow(True)
ax.legend(loc="lower right", handletextpad=0.2, borderaxespad=0.3)

# (b) panel RMSE relative to the lineage-blind design, same inclusion probabilities, adoption target
designs = [("lineage", "Lineage", C["blue"]), ("scale", "Scale", C["yellow"]), ("scale_lineage", "Scale +\nlineage", C["aqua"])]
d = by_paper[by_paper["included"] & (by_paper["target"] == "adoption")]
fig.canvas.draw()
bbox = ax2.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
allv = by_paper.loc[by_paper["included"] & (by_paper["target"] == "adoption") & (by_paper["design"] != "blind"), "ratio"]
ylo, yhi = allv.min() - 0.03, max(1.04, allv.max() + 0.03)
for x, (key, label, color) in enumerate(designs):
    v = d[d["design"] == key]["ratio"].to_numpy()
    off = swarm(v, 0.044, (yhi - ylo) / bbox.height, (len(designs) - 0.0) / bbox.width)
    ax2.scatter(x + off, v, s=5.5, color=color, alpha=0.75, lw=0, zorder=3)
    row = paper_sum[(paper_sum["target"] == "adoption") & (paper_sum["design"] == key)].iloc[0]
    ax2.plot([x - 0.29, x + 0.29], [row["median_ratio"]] * 2, color=INK, lw=1.4, zorder=4, solid_capstyle="round")
ax2.axhline(1.0, color=FAINT, lw=0.8, ls=(0, (3, 2)), zorder=1)
ax2.set_xticks(range(len(designs)))
ax2.set_xticklabels([lab for _, lab, _ in designs], linespacing=0.95)
ax2.tick_params(axis="x", length=0, pad=2)
ax2.set_xlim(-0.55, len(designs) - 0.45)
ax2.set_ylim(ylo, yhi)
ax2.set_yticks(np.arange(np.ceil(ylo * 10) / 10, yhi, 0.1).round(1))
ax2.set_ylabel("RMSE of panel means,\nrelative to lineage-blind")
ax2.set_title("(b) Spread panels recover means better", color=INK, pad=4)
ax2.grid(True, axis="y")
ax2.set_axisbelow(True)
for a_ in (ax, ax2):
    a_.tick_params(colors=INK)
save(fig, "rq4_behavior")
plt.show()


## Generated files

In [ ]:
for p in sorted(FIG.glob("*.pdf")):
    print(f"{p.name:40s} {p.stat().st_size / 1024:6.0f} KB")